# Study 08 — True-Strength · for the quants

Same seven beats, the rigorous layer: spanning-R², sign agreement, equity-curve ρ, the long/short alpha-vs-beta cut, and a White (2000) Reality Check on the TSI parameter grid. Executed on the offline synthetic universe; the headline real numbers are quoted from `../docs/results.md` (as-of + fingerprint).

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))           # study root (true_strength/ lives there)
sys.path.insert(0, os.path.abspath("../../.."))      # repo root, for quantlab
%matplotlib inline
import matplotlib.pyplot as plt
plt.rcParams["figure.figsize"] = (9.5, 5.2)
import numpy as np, pandas as pd
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")
from true_strength import data, oscillators as osc, backtest, collinearity

# Offline synthetic universe: trend/cycle names among random walks. This VALIDATES the
# machinery (oscillators agree where structure is planted). The real verdict (a liquid
# 174-name basket: NONE / MIRAGE / BUSTED) is in ../docs/results.md via verify_real.py.
frames, truth = data.synthetic_universe(seed=0)
structured = {t.ticker for t in truth}
print(f"{len(frames)} names, {len(structured)} with planted momentum structure")


20 names, 8 with planted momentum structure


## 1–3 · Claim, stakes, protocol

H₁: the TSI carries momentum information not already in MACD+RSI. Null: it is spanned by them, its position and equity curve are collinear, and any standalone Sharpe is the long bias. Pre-registered mirage line: R² ≳ 0.8, sign agreement ≳ 0.9 vs MACD, equity ρ ≳ 0.95.

Machinery sanity — the oscillators must agree more on planted structure than on noise:

In [2]:
rec = collinearity.structure_recall(frames, truth)
print(rec)

{'agree_structured': 0.846875, 'agree_noise': 0.8158333333333333}


## 4 · The Teardown

### Same shape? Level collinearity and the spanning-R²

In [3]:
print(collinearity.level_collinearity(frames).round(3).to_string())
print('\nincremental information (TSI on MACD+RSI):', 
      {k: round(v,3) if isinstance(v,float) else v for k,v in collinearity.incremental_information(frames).items()})

          n_names  median_corr    q25    q75
pair                                        
tsi~macd       20       0.9300 0.8960 0.9350
tsi~rsi        20       0.8720 0.8610 0.8890
macd~rsi       20       0.8390 0.8050 0.8620



incremental information (TSI on MACD+RSI): {'median_name_r2': 0.893, 'pooled_r2': 0.88, 'n_names': 20}


### Same position, same equity curve?

In [4]:
print('sign agreement (zero-cross):  ', {k: round(v,3) for k,v in collinearity.sign_agreement(frames,'zero').items()})
print('sign agreement (signal-cross):', {k: round(v,3) for k,v in collinearity.sign_agreement(frames,'signal').items()})
eq = collinearity.equity_collinearity(frames)
print('equity-curve correlation + Sharpes:', {k: round(v,3) for k,v in eq.items()})

sign agreement (zero-cross):   {'all_three': 0.828, 'tsi=macd': 0.986, 'tsi=rsi': 0.835, 'macd=rsi': 0.836}


sign agreement (signal-cross): {'all_three': 0.682, 'tsi=macd': 0.916, 'tsi=rsi': 0.73, 'macd=rsi': 0.719}


equity-curve correlation + Sharpes: {'corr_tsi_macd': 0.971, 'corr_tsi_rsi': 0.674, 'corr_macd_rsi': 0.684, 'sharpe_tsi': 0.12, 'sharpe_macd': 0.018, 'sharpe_rsi': -0.89}


### Alpha vs beta — the long/short timing cut

The standalone long/flat Sharpe is partly the equity risk premium (long ~half the time). Symmetrise to long/short and the unconditional drift cancels, leaving only the oscillator's *timing*. On the real run that collapses TSI 0.61 → 0.05.

In [5]:
# long/flat vs long/short Sharpe for the TSI crossover on this synthetic universe
lf = backtest.equal_weight_net(frames, lambda c: backtest.tsi_position(c, rule='signal', long_short=False))
ls = backtest.equal_weight_net(frames, lambda c: backtest.tsi_position(c, rule='signal', long_short=True))
print('long/flat  Sharpe:', round(backtest.annualized_sharpe(lf), 3))
print('long/short Sharpe:', round(backtest.annualized_sharpe(ls), 3))

long/flat  Sharpe: -0.367
long/short Sharpe: -1.037


### Reality Check on the 24-variant TSI grid

White (2000): the best-of-grid Sharpe against a mean-zero bootstrap null. On the real run p ≈ 0 — a faint, *real* generic momentum signal, which is why the verdict is **redundancy**, not noise.

In [6]:
rc = collinearity.reality_check_grid(frames, n_boot=500)
print({k: (round(v,4) if isinstance(v,float) else v) for k,v in rc.items()})

{'n_variants': 24, 'best_variant': 'tsi_40_21_13', 'best_sharpe': 0.1331, 'reality_check_pvalue': 0.626}


### Cost sweep

In [7]:
print(collinearity.cost_sweep(frames, rule='signal').round(3).to_string())

          mean_net_bps  sharpe  ann_turnover
cost_bps                                    
0               0.1250  0.0820       17.3160
5              -0.2180 -0.1420       17.3160
10             -0.5620 -0.3670       17.3160
20             -1.2490 -0.8140       17.3160
40             -2.6230 -1.7040       17.3160


### The closing argument — trade the TSI's residual over MACD+RSI

Regress the TSI out of MACD+RSI (full-sample, the generous steelman) and trade the *residual* — the part the other two can't reproduce. On the real run it earns Sharpe **−0.56**: the TSI's unique content is anti-signal, while the raw long/short TSI is already ≈ +0.05. Nothing left to be the 'true' in True Strength.

In [8]:
print({k: (round(v,4) if isinstance(v,float) else v)
       for k,v in collinearity.orthogonalised_tsi_edge(frames).items()})

{'n_names': 20, 'residual_sharpe': -1.5169, 'residual_mean_net_bps': -3.4125, 'raw_tsi_ls_sharpe': 0.1202, 'raw_tsi_ls_mean_net_bps': 0.269}


## 5–7 · Verdict, tradability, going further

**Signal `NONE`** (spanning R² 0.835, sign agreement 0.994 vs MACD, equity ρ 0.994). **Tradability `MIRAGE`** (long/short timing Sharpe 0.05 — the 0.61 was beta; the orthogonalised residual trades *negative*, −0.56). **'Truer'? `BUSTED`.** Next: extend to the wider oscillator zoo (Stochastic, CCI, %R). Numbers + fingerprint in `../docs/results.md`.